In [1]:
import os

import pandas as pd
import numpy as np
import xarray as xr

import matplotlib.pyplot as plt

from utils.cfd_wrapper import OpenFoamWrapper
from bluemath_tk.datamining import MDA

#### DATA

In [2]:
#### Data
sea_states = pd.read_csv('data/sea_states.txt',sep='\t')
sea_states.columns = ['hs', 'hs_l0', 'swl']


In [3]:
mda_ob = MDA(num_centers=10)
mda_ob.fit(data=sea_states)
sea_states_cases = mda_ob.centroids

In [4]:
sea_states_cases

,hs,hs_l0,swl
0,1.499915,0.008779,0.111836
1,0.237632,0.000590,-0.450290
2,0.466847,0.001634,0.355860
3,0.990162,0.002636,-0.478655
4,0.992268,0.002596,0.236678
5,0.626657,0.002095,-0.135218
6,0.257271,0.000721,0.015193
7,1.467712,0.008294,-0.274140
8,0.598271,0.001676,-0.561082
9,1.019005,0.004969,-0.113964


#### Inputs

In [5]:
cases_dir = 'outputs/molokai_dynamic_cases_kepsilon'
templates_dir = 'inputs/templates/molokai_kepsilon'
outputs_dir = 'outputs/molokai_dynamic_kepsilon'

In [6]:
sea_states_cases['tp'] = np.sqrt((sea_states_cases["hs"].values * 2 * np.pi) / (9.806 * sea_states_cases["hs_l0"]))
sea_states_cases['tpsoft'] = 2 * sea_states_cases['tp']
sea_states_cases['depth'] = 25 + sea_states_cases['swl']
sea_states_cases['lowfreqcutoff'] = 1 / ( 3 * sea_states_cases['tp'])
sea_states_cases['uppfreqcutoff'] = 3/  sea_states_cases['tp']

In [7]:
sea_states_cases[['hs','tp','swl']]

,hs,tp,swl
0,1.499915,10.462858,0.111836
1,0.237632,16.067961,-0.450290
2,0.466847,13.530915,0.355860
3,0.990162,15.513893,-0.478655
4,0.992268,15.648797,0.236678
5,0.626657,13.843166,-0.135218
6,0.257271,15.122787,0.015193
7,1.467712,10.648590,-0.274140
8,0.598271,15.122787,-0.561082
9,1.019005,11.462495,-0.113964


In [8]:
metamodel_parameters = sea_states_cases[['hs', 'tp', 'swl', 'tpsoft', 'depth', 'lowfreqcutoff', 'uppfreqcutoff']].to_dict(orient="list")

fixed_parameters = {'points_per_wavelenght':200,
                    'domain_lenght':2000,
                    'points_per_waveheight':15,
                    'domain_height':5,
                    #'alpha_inlet_patch_vals':[],
                    'total_run_time': 1800,
                    #'boundary_file':'/lustre/geocean/WORK/users/alonsoap/personal/estancia_NUS_2026/HyCFD/inputs/templates/openfoam/constant/polyMesh/boundary',
                    #'block_mesh_dict':'/lustre/geocean/WORK/users/alonsoap/personal/estancia_NUS_2026/HyCFD/inputs/templates/openfoam/constant/polyMesh/blockMeshDict',
                    'preprocess_script':'/lustre/geocean/WORK/users/alonsoap/personal/estancia_NUS_2026/HyCFD/inputs/scripts_openfoam/preprocess_case.sh',
                    'createmesh_script':'/lustre/geocean/WORK/users/alonsoap/personal/estancia_NUS_2026/HyCFD/inputs/scripts_openfoam/createmesh_case.sh',
                    'postprocess_script':'/lustre/geocean/WORK/users/alonsoap/personal/estancia_NUS_2026/HyCFD/inputs/scripts_openfoam/postprocess_case.sh'}

openfoam_wrapper = OpenFoamWrapper(
    templates_dir = templates_dir,
    metamodel_parameters = metamodel_parameters,
    fixed_parameters = fixed_parameters,
    output_dir = cases_dir,
)

2026-04-06 05:22:01,579 - OpenFoamWrapper - WARNING - Parameter hs is not in the default_parameters
2026-04-06 05:22:01,580 - OpenFoamWrapper - WARNING - Parameter tp is not in the default_parameters
2026-04-06 05:22:01,580 - OpenFoamWrapper - WARNING - Parameter swl is not in the default_parameters
2026-04-06 05:22:01,580 - OpenFoamWrapper - WARNING - Parameter tpsoft is not in the default_parameters
2026-04-06 05:22:01,580 - OpenFoamWrapper - WARNING - Parameter depth is not in the default_parameters
2026-04-06 05:22:01,581 - OpenFoamWrapper - WARNING - Parameter lowfreqcutoff is not in the default_parameters
2026-04-06 05:22:01,581 - OpenFoamWrapper - WARNING - Parameter uppfreqcutoff is not in the default_parameters


In [9]:
openfoam_wrapper.build_cases()

In [10]:
openfoam_wrapper.save_model(model_path=os.path.join(outputs_dir,'openfoam_model.pkl'), exclude_attributes=['_env'])